# Bronze → Silver: Transformation Exploration
## Silver Layer Design Notebook

This notebook explores the transformation logic required to convert Bronze JSON payloads
into clean, typed Parquet files in the Silver layer.

Bronze payload structure:
- `metadata` — pipeline provenance (source, dataset, execution_date, date range)
- `data.bmx.series[0].datos` — raw Banxico records

Key questions:
1. How do we list and paginate Bronze files efficiently?
2. What transformations are needed per series?
3. How do we handle empty payloads and missing values?
4. What does the final Silver schema look like?
5. What decisions do we document for `silver.py`?

In [134]:
import json
import os
import re
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.getenv("BUCKET_NAME", "banxico-pipeline-dev-datalake")
AWS_REGION  = os.getenv("AWS_REGION",  "us-east-1")

SERIES = {
    "tipo_de_cambio": {"id": "SF43718", "frequency": "daily"},
    "tiie_28":        {"id": "SF60648", "frequency": "daily"},
    "inpc":           {"id": "SP1",     "frequency": "monthly"},
}

s3_client = boto3.client("s3", region_name=AWS_REGION)

## 1. List Bronze Files

List S3 keys for a given dataset and extraction type, filtered by `execution_date`.

**Design note:** `StartAfter` uses lexicographic ordering. This works correctly
because our `execution_date` partition follows `YYYY-MM-DD` format, which sorts
chronologically as a string. Pagination handles buckets with >1000 objects.

In [135]:
def list_bronze_files(dataset: str, extraction_type: str, after_date: str) -> list[str]:
    """
    List S3 keys for Bronze payloads of a given dataset and extraction type.

    Uses StartAfter for server-side filtering — avoids downloading all keys
    and filtering client-side. Paginates automatically for buckets with >1000 objects.

    Parameters
    ----------
    dataset        : Bronze dataset name (e.g. "tipo_de_cambio")
    extraction_type: "daily" or "backfill"
    after_date     : execution_date YYYY-MM-DD — only keys after this date are returned.
                     Pass "0000-00-00" to return all files (first run).
    """
    prefix     = f"bronze/source=banxico/dataset={dataset}/extraction_type={extraction_type}"
    start_after = f"{prefix}/execution_date={after_date}"

    files              = []
    continuation_token = None

    while True:
        kwargs = dict(Bucket=BUCKET_NAME, StartAfter=start_after, Prefix=prefix)
        if continuation_token:
            kwargs["ContinuationToken"] = continuation_token

        response = s3_client.list_objects_v2(**kwargs)
        files.extend(content["Key"] for content in response.get("Contents", []))

        if not response.get("IsTruncated"):
            break

        continuation_token = response["NextContinuationToken"]

    return files

# Test: list all backfill files for inpc
files = list_bronze_files("inpc", "backfill", after_date="0000-00-00")
print(f"Found {len(files)} files")
print(files[0])  # inspect first key

Found 40 files
bronze/source=banxico/dataset=inpc/extraction_type=backfill/execution_date=2026-04-07/range_start=2023-01-01/range_end=2023-01-31/payload.json


## 2. Read Bronze Payload

Read a single Bronze payload from S3 and inspect its structure.
Each payload wraps the raw Banxico response with pipeline metadata.

In [136]:
def read_bronze_payload(s3_key: str) -> dict:
    """Read a Bronze payload from S3 and return the full dict (metadata + data)."""
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
    content  = response["Body"].read().decode("utf-8")
    return json.loads(content)

# Inspect one payload
sample_key     = files[0]
sample_payload = read_bronze_payload(sample_key)

print("── metadata ──")
print(json.dumps(sample_payload["metadata"], indent=2))

print("\n── data structure ──")
series = sample_payload["data"]["bmx"]["series"]
print(f"Number of series: {len(series)}")
print(f"Fields per record: {list(series[0]['datos'][0].keys())}")
print(f"Total records: {len(series[0]['datos'])}")
print(f"\nFirst 3 records:")
for r in series[0]["datos"][:3]:
    print(r)

── metadata ──
{
  "source": "banxico",
  "dataset": "inpc",
  "serie_id": "SP1",
  "extraction_type": "backfill",
  "execution_date": "2026-04-07",
  "execution_ts": "2026-04-07T22:56:46Z",
  "start_date": "2023-01-01",
  "end_date": "2023-01-31"
}

── data structure ──
Number of series: 1
Fields per record: ['fecha', 'dato']
Total records: 1

First 3 records:
{'fecha': '01/01/2023', 'dato': '127.336000000000'}


## 3. Parse Bronze Records

Extract raw records from `data.bmx.series[0].datos` into a DataFrame.
Metadata fields are added as columns for traceability in Silver.

**Empty payload handling:** If `datos` is missing or empty, return an empty
DataFrame — the caller decides whether to skip or raise an error.

In [137]:
def parse_bronze_records(payload: dict) -> pd.DataFrame:
    """
    Extract records from a Bronze payload into a raw DataFrame.

    Adds metadata columns (source, dataset, serie_id, processed_at) so each
    row is self-describing without needing the S3 path for context.

    Returns an empty DataFrame if the payload contains no data — caller
    decides whether to skip or raise.
    """
    series   = payload.get("data", {}).get("bmx", {}).get("series", [])
    metadata = payload.get("metadata", {})

    if not series or "datos" not in series[0]:
        print(f"WARNING: No data found | {metadata.get('start_date')} → {metadata.get('end_date')}")
        return pd.DataFrame()

    processed_at = datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rows = []

    for serie in series:
        for d in serie.get("datos", []):
            rows.append({
                "fecha":        d["fecha"],
                "dato":         d["dato"],
                "titulo":       serie["titulo"],
                "source":       metadata["source"],
                "dataset":      metadata["dataset"],
                "serie_id":     metadata["serie_id"],
                "processed_at": processed_at,
            })

    return pd.DataFrame(rows)

# Test with sample payload
df_raw = parse_bronze_records(sample_payload)
print(f"Shape: {df_raw.shape}")
print(f"Dtypes:\n{df_raw.dtypes}")
print(f"\nSample:\n{df_raw.head(3)}")

Shape: (1, 7)
Dtypes:
fecha           object
dato            object
titulo          object
source          object
dataset         object
serie_id        object
processed_at    object
dtype: object

Sample:
        fecha              dato  \
0  01/01/2023  127.336000000000   

                                              titulo   source dataset  \
0  IPC Por objeto del gasto                      ...  banxico    inpc   

  serie_id          processed_at  
0      SP1  2026-04-07T23:09:23Z  


## 4. Transform DataFrame

Apply all Silver transformations to produce a clean, typed DataFrame.

Transformations applied:
- `fecha` (dd/mm/yyyy string) → `datetime64[ns]`
- `dato` (string) → `float64` — `errors="coerce"` converts `"N/E"` to `NaN`
- `titulo` — strip leading/trailing whitespace + collapse internal spaces
- Column rename to final Silver schema

**N/E handling:** Banxico uses `"N/E"` for unavailable data points. `pd.to_numeric`
with `errors="coerce"` converts these to `NaN` automatically — no explicit handling needed.

**Weekend behavior (SF61745):** The target rate series includes weekends with the same
value as the preceding Friday. This reflects actual monetary policy — the rate does not
change on weekends. Weekend rows are preserved in Silver.

In [138]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply all Silver transformations to a raw Bronze DataFrame.

    - Parses Banxico date format (dd/mm/yyyy) to datetime64
    - Casts value strings to float64 (N/E → NaN via coerce)
    - Normalizes title whitespace
    - Renames columns to final Silver schema
    """
    df = df.copy()

    df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
    df["dato"]  = pd.to_numeric(df["dato"],   errors="coerce")
    df["titulo"] = df["titulo"].str.strip().apply(lambda x: re.sub(r"\s+", " ", x))

    df = df.rename(columns={
        "fecha":  "date",
        "dato":   "value",
        "titulo": "title",
    })

    return df[["date", "source", "dataset", "serie_id", "processed_at", "value", "title"]]

df_silver = transform(df_raw)

print(f"Shape: {df_silver.shape}")
print(f"\nDtypes:\n{df_silver.dtypes}")
print(f"\nNull values:\n{df_silver.isnull().sum()}")
print(f"\nSample:\n{df_silver.head(5)}")

Shape: (1, 7)

Dtypes:
date            datetime64[ns]
source                  object
dataset                 object
serie_id                object
processed_at            object
value                  float64
title                   object
dtype: object

Null values:
date            0
source          0
dataset         0
serie_id        0
processed_at    0
value           0
title           0
dtype: int64

Sample:
        date   source dataset serie_id          processed_at    value  \
0 2023-01-01  banxico    inpc      SP1  2026-04-07T23:09:23Z  127.336   

                                               title  
0  IPC Por objeto del gasto Nacional I n d i c e ...  


## 5. Data Quality Checks

Verify the transformed DataFrame before writing to Silver.
These checks inform the Great Expectations suite in P3.

In [140]:
print("── Date range ──")
print(f"Min: {df_silver['date'].min().date()}")
print(f"Max: {df_silver['date'].max().date()}")

print("\n── Null values ──")
print(df_silver.isnull().sum())

print("\n── Value stats ──")
print(df_silver["value"].describe())

print("\n── Duplicate dates ──")
dupes = df_silver[df_silver.duplicated(subset=["date"], keep=False)]
print(f"Duplicate date rows: {len(dupes)}")
if len(dupes) > 0:
    print(dupes)

── Date range ──
Min: 2023-01-01
Max: 2023-01-01

── Null values ──
date            0
source          0
dataset         0
serie_id        0
processed_at    0
value           0
title           0
dtype: int64

── Value stats ──
count      1.000
mean     127.336
std          NaN
min      127.336
25%      127.336
50%      127.336
75%      127.336
max      127.336
Name: value, dtype: float64

── Duplicate dates ──
Duplicate date rows: 0


## 6. Silver Layer — Transformation Decisions

Based on this exploration, the following decisions are implemented in `silver.py`:

| Decision | Observation | Action |
|---|---|---|
| Date parsing | Banxico returns `dd/mm/yyyy` strings | Parse to `datetime64[ns]` |
| Missing values | `"N/E"` used for unavailable data | `pd.to_numeric(errors="coerce")` → `NaN` |
| Value type | All values returned as strings | Cast to `float64` |
| Title cleanup | Irregular whitespace in `titulo` field | `str.strip()` + `re.sub(r'\s+', ' ')` |
| Weekend rows (SF60648) | TIIE 28 includes all calendar days | Preserved — rate is valid on weekends |
| Empty payloads | Some periods may have no data | Return empty DataFrame, log warning, skip |
| Column schema | Consistent across all three series | `date, source, dataset, serie_id, processed_at, value, title` |
| Deduplication | Daily runs have 7-day rolling overlap | Deduplicate by `(serie_id, date)`, keep latest `processed_at` |

## Next Steps
- [ ] Implement `silver.py` with `process_dataset()` and `run_silver()`
- [ ] Write Parquet partitioned by `year/month` to Silver layer
- [ ] Integrate checkpoint store for incremental processing
- [ ] Define Great Expectations suite based on value ranges observed here